# Support Vector Regression (SVR) — Complete Notes

---

## 1. Background: From Linear Regression to SVR

Before diving into SVR, recall what **ordinary linear regression** does.

Given a dataset of $n$ points $\{(x_i, y_i)\}_{i=1}^{n}$, linear regression finds a function:

$$f(x) = w^T x + b$$

by minimizing the **sum of squared errors**:

$$\mathcal{L}_{\text{OLS}} = \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2$$

**Where:**
- $x_i \in \mathbb{R}^d$ — the $i$-th input feature vector (a column vector of $d$ features)
- $y_i \in \mathbb{R}$ — the true target value for sample $i$
- $\hat{y}_i = f(x_i) = w^T x_i + b$ — the predicted value
- $w \in \mathbb{R}^d$ — the weight vector (one weight per feature)
- $b \in \mathbb{R}$ — the bias (intercept) term

**Problem with least squares:**
- Every single prediction error is penalized, no matter how small
- Squaring the errors makes the model **highly sensitive to outliers** — one bad point can pull the entire regression line significantly

**SVR's key insight:** *"If the prediction is close enough, don't penalize it at all."*

This introduces the concept of an **$\varepsilon$-insensitive loss**.

---

## 2. The $\varepsilon$-Insensitive Loss Function

Instead of penalizing every error, SVR defines a **tube of width $\varepsilon$** around the regression function. Errors smaller than $\varepsilon$ are completely ignored.

The $\varepsilon$-insensitive loss is:

$$L_\varepsilon(y, f(x)) = \max\left(0,\ |y - f(x)| - \varepsilon\right)$$

**Where:**
- $\varepsilon > 0$ — a hyperparameter you choose; it defines the "acceptable error" margin
- $y$ — true target value
- $f(x)$ — predicted value
- $|y - f(x)|$ — the absolute prediction error

**Interpretation:**
- If $|y - f(x)| \leq \varepsilon$ → loss is $0$ (point is inside the tube, no penalty)
- If $|y - f(x)| > \varepsilon$ → loss is $|y - f(x)| - \varepsilon$ (only the excess beyond $\varepsilon$ is penalized)

This is fundamentally different from squared error, which penalizes *every* deviation.

---

## 3. The Geometric Picture: The $\varepsilon$-Tube

Imagine drawing the regression line $f(x) = w^T x + b$, then drawing two parallel lines at distance $\varepsilon$ above and below it. This creates a **tube of width $2\varepsilon$**:

$$\text{Upper boundary: } f(x) + \varepsilon$$
$$\text{Lower boundary: } f(x) - \varepsilon$$

- **Points inside the tube** → residual $\leq \varepsilon$ → **zero loss**, completely ignored
- **Points outside the tube** → residual $> \varepsilon$ → penalized by how far they are beyond the boundary
- **Points on or touching the tube boundary** → these are the critical points, called **Support Vectors**

The model *only depends on the support vectors*. All other points are irrelevant to the final solution. This is what makes SVR **sparse** and **robust**.

---

## 4. The Primal Optimization Problem

SVR seeks a function $f(x) = w^T x + b$ that:
1. Is as **flat** as possible (simple, low complexity)
2. Has prediction errors within $\varepsilon$ for all training points

**Why flatness?** A flat function (small $\|w\|$) generalizes better. It's the same regularization idea as in Ridge regression.

### Step 1: Objective — Maximize Flatness

$$\min_{w,\, b} \quad \frac{1}{2} \|w\|^2$$

**Where:**
- $\|w\|^2 = w^T w = \sum_{j=1}^{d} w_j^2$ — the squared Euclidean norm of the weight vector
- Minimizing $\|w\|^2$ forces $w$ to be small → the function is flat / smooth

### Step 2: Constraints — Stay Inside the Tube

$$y_i - w^T x_i - b \leq \varepsilon \quad \text{(prediction not too low)}$$
$$w^T x_i + b - y_i \leq \varepsilon \quad \text{(prediction not too high)}$$

These two constraints together enforce $|y_i - f(x_i)| \leq \varepsilon$ for all $i$.

### Step 3: Allow Violations with Slack Variables

In practice, it may be impossible to fit all points within the tube. We introduce **slack variables** to allow violations:

$$\xi_i \geq 0 \quad \text{(upper slack — for points above the tube)}$$
$$\xi_i^* \geq 0 \quad \text{(lower slack — for points below the tube)}$$

The relaxed constraints become:

$$y_i - w^T x_i - b \leq \varepsilon + \xi_i$$
$$w^T x_i + b - y_i \leq \varepsilon + \xi_i^*$$
$$\xi_i,\ \xi_i^* \geq 0$$

**Interpretation:**
- $\xi_i > 0$ means the $i$-th point lies **above** the upper tube boundary by $\xi_i$
- $\xi_i^* > 0$ means the $i$-th point lies **below** the lower tube boundary by $\xi_i^*$
- A point can only violate one boundary at a time, so at most one of $\xi_i, \xi_i^*$ is nonzero for any given $i$

### Step 4: Full Primal Problem

$$\min_{w,\, b,\, \xi_i,\, \xi_i^*} \quad \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*)$$

subject to:

$$y_i - w^T x_i - b \leq \varepsilon + \xi_i$$
$$w^T x_i + b - y_i \leq \varepsilon + \xi_i^*$$
$$\xi_i,\ \xi_i^* \geq 0 \quad \forall\, i = 1, \ldots, n$$

**Where:**
- $\frac{1}{2}\|w\|^2$ — flatness/regularization term
- $C > 0$ — the **regularization parameter** (hyperparameter you tune)
  - **Large $C$** → heavily penalize violations → model tries hard to fit every point → risk of overfitting
  - **Small $C$** → allow more violations → smoother function → risk of underfitting
- $\sum_{i=1}^{n}(\xi_i + \xi_i^*)$ — total amount of constraint violation across all training points

This is a **convex quadratic programming (QP)** problem — it has a unique global minimum.

---

## 5. The Lagrangian and Dual Formulation

To solve this efficiently and enable the **kernel trick**, we convert to the **dual problem** using Lagrange multipliers.

### Setting Up the Lagrangian

We attach a non-negative Lagrange multiplier to each constraint:

- $\alpha_i \geq 0$ for the constraint $y_i - w^T x_i - b \leq \varepsilon + \xi_i$
- $\alpha_i^* \geq 0$ for the constraint $w^T x_i + b - y_i \leq \varepsilon + \xi_i^*$
- $\eta_i \geq 0$ for the constraint $\xi_i \geq 0$
- $\eta_i^* \geq 0$ for the constraint $\xi_i^* \geq 0$

The full Lagrangian is:

$$\mathcal{L} = \frac{1}{2}\|w\|^2 + C\sum_{i=1}^n(\xi_i + \xi_i^*) - \sum_{i=1}^n(\eta_i \xi_i + \eta_i^* \xi_i^*)$$
$$- \sum_{i=1}^n \alpha_i \left(\varepsilon + \xi_i - y_i + w^T x_i + b\right) - \sum_{i=1}^n \alpha_i^* \left(\varepsilon + \xi_i^* + y_i - w^T x_i - b\right)$$

### KKT Stationarity Conditions

Taking partial derivatives and setting to zero:

$$\frac{\partial \mathcal{L}}{\partial w} = 0 \implies w = \sum_{i=1}^n (\alpha_i - \alpha_i^*)\, x_i$$

$$\frac{\partial \mathcal{L}}{\partial b} = 0 \implies \sum_{i=1}^n (\alpha_i - \alpha_i^*) = 0$$

$$\frac{\partial \mathcal{L}}{\partial \xi_i} = 0 \implies C - \alpha_i - \eta_i = 0 \implies \alpha_i \in [0, C]$$

$$\frac{\partial \mathcal{L}}{\partial \xi_i^*} = 0 \implies C - \alpha_i^* - \eta_i^* = 0 \implies \alpha_i^* \in [0, C]$$

**Key insight from the first equation:** The weight vector $w$ is expressed entirely as a **linear combination of training data points**, weighted by $(\alpha_i - \alpha_i^*)$.

### The Dual Problem

Substituting the KKT conditions back into the Lagrangian and simplifying (the algebra eliminates $w$, $b$, $\xi_i$, $\xi_i^*$), we obtain the dual problem:

$$\max_{\alpha_i,\, \alpha_i^*} \quad -\frac{1}{2} \sum_{i=1}^{n}\sum_{j=1}^{n} (\alpha_i - \alpha_i^*)(\alpha_j - \alpha_j^*)\, x_i^T x_j - \varepsilon \sum_{i=1}^{n}(\alpha_i + \alpha_i^*) + \sum_{i=1}^{n} y_i (\alpha_i - \alpha_i^*)$$

subject to:

$$\sum_{i=1}^{n} (\alpha_i - \alpha_i^*) = 0$$
$$0 \leq \alpha_i,\ \alpha_i^* \leq C \quad \forall\, i$$

**Where:**
- $\alpha_i, \alpha_i^* \geq 0$ — **dual variables** (Lagrange multipliers), one pair per training point
- $x_i^T x_j$ — dot product between training samples $i$ and $j$ (this is what the kernel will replace!)
- The first term penalizes complexity in dual space
- The second term penalizes the width of the tube (controlled by $\varepsilon$)
- The third term rewards fitting the targets

### Prediction Function (Dual Form)

Since $w = \sum_{i=1}^n (\alpha_i - \alpha_i^*) x_i$, the prediction becomes:

$$f(x) = w^T x + b = \sum_{i=1}^{n} (\alpha_i - \alpha_i^*)\, x_i^T x + b$$

**Crucially:** By the KKT complementary slackness conditions, most $(\alpha_i - \alpha_i^*)$ will be exactly **zero**. Only points on or outside the tube boundary have nonzero $\alpha_i$ or $\alpha_i^*$ — these are the **Support Vectors**. The sum above runs over only a sparse subset of training points.

---

## 6. Why "Support Vectors"?

The name comes from the dual solution. After solving, define:

$$\beta_i = \alpha_i - \alpha_i^*$$

Then:

$$f(x) = \sum_{i \in SV} \beta_i\, x_i^T x + b$$

where $SV$ is the set of **support vectors** — training points with $\beta_i \neq 0$.

By complementary slackness:
- Points **strictly inside** the tube: $\alpha_i = \alpha_i^* = 0$ → $\beta_i = 0$ → **not support vectors**
- Points **on the tube boundary**: $0 < \alpha_i < C$ or $0 < \alpha_i^* < C$ → $\beta_i \neq 0$ → **support vectors**
- Points **outside the tube**: $\alpha_i = C$ or $\alpha_i^* = C$ → **support vectors** (with slack)

The model is literally "supported" by these vectors — remove any non-support-vector, and the model is unchanged.

---

## 7. The Kernel Trick — Handling Nonlinearity

Look at the dual problem and the prediction function. The input data only appears as **dot products**:

$$x_i^T x_j \quad \text{(in the dual)} \qquad \text{and} \qquad x_i^T x \quad \text{(in prediction)}$$

The **kernel trick** replaces these dot products with a **kernel function** $K(x_i, x_j)$:

$$K(x_i, x_j) = \phi(x_i)^T \phi(x_j)$$

where $\phi: \mathbb{R}^d \to \mathcal{H}$ is a (possibly infinite-dimensional) feature map. We **never need to compute $\phi$ explicitly** — we only need to evaluate $K$.

### Kernelized Dual Problem

$$\max_{\alpha,\alpha^*} \quad -\frac{1}{2}\sum_{i,j}(\alpha_i - \alpha_i^*)(\alpha_j - \alpha_j^*) K(x_i, x_j) - \varepsilon\sum_i(\alpha_i + \alpha_i^*) + \sum_i y_i(\alpha_i - \alpha_i^*)$$

### Kernelized Prediction

$$\boxed{f(x) = \sum_{i \in SV} (\alpha_i - \alpha_i^*)\, K(x_i, x) + b}$$

### Common Kernels

| Kernel | Formula | Notes |
|--------|---------|-------|
| **Linear** | $K(x_i, x) = x_i^T x$ | No transformation; same as primal |
| **Polynomial** | $K(x_i, x) = (x_i^T x + c)^p$ | Degree $p$, coefficient $c > 0$ |
| **RBF / Gaussian** | $K(x_i, x) = \exp\!\left(-\gamma \|x_i - x\|^2\right)$ | Most widely used; infinite-dimensional feature space |
| **Sigmoid** | $K(x_i, x) = \tanh(\kappa\, x_i^T x + \theta)$ | Neural network-like |

**Where (for RBF):**
- $\gamma > 0$ — controls the "width" of the Gaussian; larger $\gamma$ → narrower kernel → more local/complex model

---

## 8. Hyperparameters Summary

| Parameter | Symbol | Role | Effect |
|-----------|--------|------|--------|
| Regularization | $C$ | Trade-off between flatness and fitting | ↑$C$: less regularization, fits tighter; ↓$C$: smoother |
| Tube width | $\varepsilon$ | Size of the insensitive zone | ↑$\varepsilon$: more tolerance, fewer SVs, sparser model |
| Kernel bandwidth | $\gamma$ | RBF kernel width | ↑$\gamma$: complex, local; ↓$\gamma$: smooth, global |
| Kernel type | — | Linear, RBF, Poly, ... | Determines feature space geometry |

---

## 9. Computing the Bias Term $b$

After solving for $\alpha_i, \alpha_i^*$, we compute $b$ using any support vector $i$ that lies strictly on the tube boundary (i.e., $0 < \alpha_i < C$ or $0 < \alpha_i^* < C$):

- If $0 < \alpha_i < C$: the point is on the **upper** boundary, so $y_i - f(x_i) = \varepsilon$:
$$b = y_i - \sum_{j \in SV} (\alpha_j - \alpha_j^*) K(x_j, x_i) - \varepsilon$$

- If $0 < \alpha_i^* < C$: the point is on the **lower** boundary, so $f(x_i) - y_i = \varepsilon$:
$$b = y_i - \sum_{j \in SV} (\alpha_j - \alpha_j^*) K(x_j, x_i) + \varepsilon$$

In practice, $b$ is often computed as the average over all such boundary support vectors for numerical stability.

---

## 10. SVR vs Linear Regression — Side by Side

| Aspect | Linear Regression (OLS) | SVR |
|--------|------------------------|-----|
| **Loss function** | $\sum (y_i - \hat{y}_i)^2$ | $\varepsilon$-insensitive: ignore if within $\varepsilon$ |
| **Outlier sensitivity** | High (squared error amplifies outliers) | Lower (bounded penalty beyond $\varepsilon$) |
| **Kernel / Nonlinearity** | No | Yes, via kernel trick |
| **Sparsity** | No (uses all points) | Yes (only support vectors) |
| **Interpretability** | High ($w$ directly shows feature importance) | Low (opaque for nonlinear kernels) |
| **Computational cost** | $O(nd)$ | $O(n^2)$ to $O(n^3)$ |
| **Tuning burden** | Low (just 1 parameter) | High ($C$, $\varepsilon$, kernel, $\gamma$) |

---

## 11. When to Use SVR

**Use SVR when:**
- Dataset is **small to medium** (hundreds to low thousands of rows)
- Data has **outliers** you want to be robust against
- The relationship is **nonlinear** (use RBF kernel)
- You want a **sparse model** that depends only on key data points
- High-dimensional feature spaces (SVR handles these well)

**Avoid SVR when:**
- Dataset is **very large** (SVR's $O(n^2)$–$O(n^3)$ complexity makes it infeasible for $n > 10{,}000$)
- You need fast training or online learning
- You need an easily interpretable model
- The relationship is straightforward and linear (just use OLS)

---

## 12. Quick Recap — The Full SVR Roadmap

$$\underbrace{\text{Primal: } \min \frac{1}{2}\|w\|^2 + C\sum(\xi_i + \xi_i^*)}_{\text{flatness + penalized violations}}$$

$$\Downarrow \text{ Lagrangian + KKT conditions}$$

$$\underbrace{\text{Dual: } \max \left[-\frac{1}{2}\sum_{i,j}(\alpha_i - \alpha_i^*)(\alpha_j - \alpha_j^*)K(x_i,x_j) - \varepsilon\sum(\alpha_i+\alpha_i^*) + \sum y_i(\alpha_i - \alpha_i^*)\right]}_{\text{solve for } \alpha_i, \alpha_i^*}$$

$$\Downarrow \text{ plug into prediction}$$

$$\underbrace{f(x) = \sum_{i \in SV}(\alpha_i - \alpha_i^*) K(x_i, x) + b}_{\text{sparse: only support vectors contribute}}$$

> **Key insight:** SVR is SVM with a different loss function. Everything else — the convex optimization, Lagrangian duality, kernel trick, and sparsity — is identical in philosophy.